In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Finding best numerical feature set

In [83]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

numeric_candidates = pd.DataFrame(anime_data.values())

statistics_df = pd.json_normalize(numeric_candidates["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

numeric_candidates = pd.concat(
    [numeric_candidates.drop(columns=["statistics"]), statistics_df],
    axis=1,
)

numeric_columns = [
    "mean",
    "rank",
    "popularity",
    "num_list_users",
    "num_scoring_users",
    "num_episodes",
    "statistics_num_list_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

numeric_analysis_df = (
    numeric_candidates[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

numeric_analysis_df.head()

,mean,rank,popularity,num_list_users,num_scoring_users,num_episodes,statistics_num_list_users,watching,completed,on_hold,dropped,plan_to_watch
0,8.89,26.0,118,1314117,505249,74,1313880,177147,501891,94398,43705,496739
1,8.12,549.0,1195,234811,80450,26,234759,14043,94627,9711,6501,109877
2,9.11,3.0,3,3659053,2298157,64,3658804,285524,2645506,120875,64923,541976
3,9.03,10.0,8,3164257,1965469,148,3164039,381795,2174185,151790,69066,387203
4,8.24,385.0,270,819854,289082,25,819715,56032,322940,37493,36737,366513


In [166]:
def calculate_vif(df):
    rows = []
    for target_col in df.columns:
        X = df.drop(columns=[target_col]).to_numpy(dtype=float)
        y = df[target_col].to_numpy(dtype=float)

        model = LinearRegression()
        model.fit(X, y)
        r_squared = model.score(X, y)

        vif = np.inf if np.isclose(1 - r_squared, 0) else 1 / (1 - r_squared)
        rows.append({
            "feature": target_col,
            "r_squared_from_other_features": r_squared,
            "vif": vif,
        })

    return pd.DataFrame(rows).sort_values("vif", ascending=False)

reduced_numeric_analysis_df = numeric_analysis_df.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        "watching",
        # "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        # "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",
)

vif_results = calculate_vif(reduced_numeric_analysis_df)
vif_results

,feature,r_squared_from_other_features,vif
1,popularity,0.338158,1.510934
3,dropped,0.253840,1.340195
0,mean,0.122581,1.139707
2,num_episodes,0.087032,1.095329


Build features

In [167]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df_num = builder.build_num_features().set_index("id")
anime_genres_df = builder.build_genre_features().set_index("anime_id")
anime_studios_df = builder.build_studio_features().set_index("anime_id")
synopsis_tfidf, synopsis_features = builder.build_synopsis_features()
synopsis_svd_df = builder.apply_svd(synopsis_tfidf, synopsis_features.index)

anime_df_complete = pd.concat([
    anime_df_num,
    anime_genres_df,
    synopsis_svd_df,
], axis=1).dropna()

builder.svd_explained_variance

np.float64(0.41006426159563164)

In [177]:
anime_df_num_new = pd.DataFrame(anime_data.values())
anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "main_picture",
        "title",
        "synopsis",
        "media_type",
        "status",
        "genres",
        "rating",
        "recommendations",
        "studios",
    ],
    errors="ignore",
)

statistics_df = pd.json_normalize(anime_df_num_new["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

anime_df_num_new = pd.concat([anime_df_num_new.drop(columns=["statistics"]), statistics_df], axis=1)

anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        "watching",
        # "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        # "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",).set_index("id").dropna()

anime_df_num_new

,mean,popularity,num_episodes,dropped
id,,,,
19,8.89,118,74,43705
1827,8.12,1195,26,6501
5114,9.11,3,64,64923
11061,9.03,8,148,69066
13125,8.24,270,25,36737
...,...,...,...,...
16512,6.83,1602,13,14122
873,6.86,2826,26,4105
32032,6.71,2230,12,9280


In [178]:
# Pick the feature set to use below.
# anime_df = anime_df_complete
anime_df = pd.concat([anime_df_num_new, anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_complete, anime_studios_df], axis=1).dropna()
# anime_df = anime_df_num.dropna()
# anime_df = anime_genres_df.dropna()
# anime_df = anime_studios_df.dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df, anime_studios_df], axis=1).dropna()

Convert each anime in df to vectors

In [ ]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

anime_df_scaled.head()

Get user Data

In [180]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Hit Rate

In [ ]:
from anime_evaluation import HitRateEvaluator

n_runs = 50
result_top_ks = (5, 10)
uncertainty_weight = 7.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = evaluator.tune_bayesian_uncertainty(
    weights=[uncertainty_weight],
    n_runs=n_runs,
    top_ks=result_top_ks,
    random_state=42,
)

if baseline_summary is not None and not baseline_summary.empty:
    average_metrics = bayesian_summary.merge(
        baseline_summary,
        on="k",
        how="left",
    )
else:
    average_metrics = bayesian_summary.copy()

average_metrics = average_metrics.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,7.5,5,0.602,0.222919,0.097097,0.035955,3.01,0.098,0.128692,0.015806,0.020757,0.49
1,7.5,10,0.361,0.128625,0.116452,0.041492,3.61,0.066,0.062312,0.021290,0.020101,0.66


## Numeric Combo Sweep

This sweep keeps the core numeric signals (`mean`, `popularity`, `num_episodes`) and tests all 64 combinations of optional activity/support signals inside the full feature context: numeric combo + genres + synopsis SVD. Use a smaller run count for the sweep, then rerun the top few combos with more runs.

In [187]:
from itertools import combinations
from anime_evaluation import HitRateEvaluator
from anime_recommender import SimilarityRecommender

core_numeric_features = ["mean", "popularity", "num_episodes"]
optional_numeric_features = [
    "num_scoring_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

numeric_combo_base = numeric_candidates[["id"] + numeric_columns].copy()
for column in numeric_columns:
    numeric_combo_base[column] = pd.to_numeric(
        numeric_combo_base[column],
        errors="coerce",
    )
numeric_combo_base = numeric_combo_base.set_index("id").dropna()

combo_n_runs = 20
combo_top_ks = (5, 10)
combo_uncertainty_weight = 7.5
combo_rows = []

for combo_size in range(len(optional_numeric_features) + 1):
    for optional_combo in combinations(optional_numeric_features, combo_size):
        selected_numeric_features = core_numeric_features + list(optional_combo)
        feature_set_name = "core"
        if optional_combo:
            feature_set_name += "_" + "_".join(optional_combo)

        combo_anime_df = pd.concat(
            [
                numeric_combo_base[selected_numeric_features],
                anime_genres_df,
                synopsis_svd_df,
            ],
            axis=1,
        ).dropna()

        combo_recommender = SimilarityRecommender()
        combo_recommender.create_anime_vectors(combo_anime_df)

        combo_evaluator = HitRateEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )

        _, combo_summary, _, _, combo_baseline_summary = (
            combo_evaluator.tune_bayesian_uncertainty(
                weights=[combo_uncertainty_weight],
                n_runs=combo_n_runs,
                top_ks=combo_top_ks,
                random_state=42,
            )
        )

        if combo_baseline_summary is not None and not combo_baseline_summary.empty:
            combo_summary = combo_summary.merge(
                combo_baseline_summary,
                on="k",
                how="left",
            )

        for row in combo_summary.to_dict("records"):
            row["feature_set"] = feature_set_name
            row["numeric_features"] = ", ".join(selected_numeric_features)
            row["n_numeric_features"] = len(selected_numeric_features)
            row["n_runs"] = combo_n_runs
            combo_rows.append(row)

numeric_combo_results = pd.DataFrame(combo_rows)
numeric_combo_summary = numeric_combo_results.sort_values(
    ["k", "avg_precision_at_k", "avg_hit_rate"],
    ascending=[True, False, False],
)

numeric_combo_summary.head(20)


KeyboardInterrupt: 

## Results

The best-performing feature set was **all_features**, which combines the original numeric metadata, genre one-hot features, and synopsis SVD features. The saved feature-set checks use **50 repeated holdout runs** with Bayesian Ridge and an uncertainty weight of **7.5**. Adding studio one-hot features did not improve the selected representation, and the VIF-reduced activity numeric set also performed worse than the original numeric set.

| Feature set | P@5 | P@10 | Improvement over baseline |
| --- | ---: | ---: | ---: |
| all_features | 0.528 | 0.314 | 4.55x / 4.13x |
| all_features_studios | 0.460 | 0.314 | 3.97x / 4.13x |
| numeric_genres_studios | 0.420 | 0.290 | 3.62x / 3.82x |
| numeric_genres | 0.400 | 0.264 | 3.45x / 3.47x |
| numeric_svd | 0.240 | 0.278 | 2.07x / 3.66x |
| activity_numeric_genres_svd | 0.244 | 0.210 | 2.10x / 2.76x |
| only_numeric | 0.196 | 0.156 | 1.69x / 2.05x |
| genres_studios | 0.088 | 0.138 | n/a |
| numeric_svd_studios | 0.092 | 0.092 | 0.79x / 1.21x |
| only_genres | 0.044 | 0.050 | n/a |
| genres_svd | 0.040 | 0.042 | n/a |

The selected-feature results above were saved in `metrics/features_set_selection_20260615_153918.csv`. Baseline improvement is listed as `n/a` when the active feature dataframe did not include `mean`, so the global-mean baseline was not computed for that run. The activity numeric set passed the VIF check, but VIF only measures multicollinearity; it did not translate into better recommendation quality.

Conclusion: keep **all_features** as the selected feature representation for the recommender. The original numeric feature set was initially chosen by intuition and domain preference, but the evaluation confirms it works better than the VIF-inspired activity numeric set for this recommender.